# 🧬 DEMO: Minimum Edit Distance (Khoảng cách chỉnh sửa tối thiểu)

Notebook này minh hoạ **mọi trường hợp** của bài toán trong mục 2.9.1 (Jurafsky & Martin, *Speech and Language Processing*):

| Phần | Nội dung |
|---|---|
| 1 | Ba phép chỉnh sửa: chèn / xoá / thay thế + đường đi `intention → execution` |
| 2 | Vì sao không thể tìm kiếm ngây thơ → cần **quy hoạch động** |
| 3 | Công thức D[i,j] (Eq. 2.19, 2.20) và cài đặt thuật toán Fig. 2.21 |
| 4 | Bảng khoảng cách của `intention` ↔ `execution` (Fig. 2.20) |
| 5 | Tính tay một ô để hiểu công thức |
| 6 | Các trường hợp biên (chuỗi rỗng, giống nhau, khác hoa/thường, tiếng Việt…) |
| 7 | Các biến thể chi phí (Levenshtein 1-1-1, 1-1-2, trọng số tuỳ ý, theo bàn phím) |
| 8 | **Alignment**: backpointer + backtrace (Fig. 2.22), nhiều alignment tối ưu |
| 9 | Kiểm chứng tính đúng đắn (áp dụng ops, so với brute-force) |
| 10 | Tối ưu: bộ nhớ O(m), ngưỡng k |
| 11 | Ứng dụng: sửa lỗi chính tả, Word Error Rate, tìm alignment xác suất (gợi ý Viterbi) |
| 12 | Liên hệ với LCS |

> 💡 Chạy tuần tự từ trên xuống (Shift+Enter). Chỉ cần Python 3, không cần thư viện ngoài.

## 1. Ba phép chỉnh sửa và đường đi `intention → execution`

- **Insertion (chèn)**: thêm 1 ký tự vào chuỗi nguồn.
- **Deletion (xoá)**: bỏ 1 ký tự khỏi chuỗi nguồn.
- **Substitution (thay thế)**: đổi 1 ký tự thành ký tự khác.

Hình 2.19 trong sách: `intention` → `execution` đi qua 5 bước. Ta mô phỏng lại từng bước bằng code.

In [ ]:
def apply_step(s, op, pos, ch=None):
    """Áp dụng 1 phép chỉnh sửa lên chuỗi s (pos tính từ 0)."""
    if op == "del":   return s[:pos] + s[pos+1:]
    if op == "ins":   return s[:pos] + ch + s[pos:]
    if op == "sub":   return s[:pos] + ch + s[pos+1:]

s = "intention"
steps = [("del", 0, None, "delete i"),
         ("sub", 0, "e",  "substitute n by e"),
         ("sub", 1, "x",  "substitute t by x"),
         ("ins", 4, "u",  "insert u"),
         ("sub", 3, "c",  "substitute n by c")]
print(f"{'':28s}{s}")
for op, pos, ch, desc in steps:
    s = apply_step(s, op, pos, ch)
    print(f"{desc:28s}{s}")
assert s == "execution"

Nếu mỗi phép **chèn/xoá tốn 1** và **thay thế tốn 2**: 1 (del) + 2 + 2 + 1 (ins) + 2 = **8**.
Nếu thay thế chỉ tốn 1 thì tổng = 1 + 1 + 1 + 1 + 1 = **5**.

## 2. Vì sao cần quy hoạch động?

Cách ngây thơ: thử đệ quy 3 nhánh (xoá, chèn, thay thế) tại mỗi bước. Số trạng thái **bùng nổ**, vì rất nhiều đường đi khác nhau lại dẫn tới **cùng một trạng thái** (cùng cặp `(i, j)`). Quy hoạch động chỉ **nhớ** lời giải mỗi trạng thái một lần.

In [ ]:
import time

def naive(i, j, s, t, counter):
    counter[0] += 1                      # đếm số lần gọi hàm
    if i == 0: return j                  # cần chèn j ký tự
    if j == 0: return i                  # cần xoá i ký tự
    sc = 0 if s[i-1] == t[j-1] else 2
    return min(naive(i-1, j,   s, t, counter) + 1,
               naive(i,   j-1, s, t, counter) + 1,
               naive(i-1, j-1, s, t, counter) + sc)

def memo_count(s, t):
    cache, calls = {}, [0]
    def go(i, j):
        calls[0] += 1
        if (i, j) in cache: return cache[(i, j)]
        if i == 0: r = j
        elif j == 0: r = i
        else:
            sc = 0 if s[i-1] == t[j-1] else 2
            r = min(go(i-1, j)+1, go(i, j-1)+1, go(i-1, j-1)+sc)
        cache[(i, j)] = r
        return r
    return go(len(s), len(t)), len(cache), calls[0]

for s, t in [("intent", "exec"), ("intenti", "execu"), ("intention", "execution")]:
    c = [0]; t0 = time.time()
    r = naive(len(s), len(t), s, t, c)
    t1 = time.time() - t0
    r2, states, calls = memo_count(s, t)
    print(f"{s:>9} -> {t:<9} D={r}  | đệ quy thuần: {c[0]:>9,} lần gọi ({t1:.3f}s)"
          f"  | có nhớ: {states} trạng thái duy nhất = (n+1)(m+1)")

Với 9 ký tự: ~2 triệu lần gọi so với chỉ **100 trạng thái** (10 × 10). Khi chuỗi dài hơn, sự chênh lệch là **số mũ** so với **bậc hai**.

**Tính chất cấu trúc con tối ưu** (giải thích trong sách): nếu chuỗi `exention` nằm trên đường tối ưu `intention → execution`, thì đoạn `intention → exention` cũng phải tối ưu. Nếu có đoạn ngắn hơn, ta thay vào và được đường tổng ngắn hơn — mâu thuẫn.

## 3. Thuật toán Min-Edit-Distance (Fig. 2.21)

Ký hiệu: nguồn `X` dài `n`, đích `Y` dài `m`, `D[i, j]` = khoảng cách giữa `i` ký tự đầu của X và `j` ký tự đầu của Y.

- **Khởi tạo**: `D[i,0] = i` (xoá i ký tự), `D[0,j] = j` (chèn j ký tự).
- **Truy hồi (Eq. 2.19)**: `D[i,j] = min( D[i-1,j] + del, D[i,j-1] + ins, D[i-1,j-1] + sub )`.
- **Kết quả**: `D[n, m]`.

Ta viết bản **tổng quát**: chi phí là *hàm* (có thể phụ thuộc vào từng ký tự).

In [ ]:
def uniform(ins=1, dele=1, sub=2):
    """Tạo 3 hàm chi phí cố định. Thay thế ký tự giống nhau luôn tốn 0."""
    return (lambda c: ins,
            lambda c: dele,
            lambda a, b: 0 if a == b else sub)

def edit_matrix(source, target, ins_cost=None, del_cost=None, sub_cost=None):
    """Trả về ma trận D kích thước (n+1) x (m+1)."""
    d0 = uniform()
    ins_cost = ins_cost or d0[0]
    del_cost = del_cost or d0[1]
    sub_cost = sub_cost or d0[2]
    n, m = len(source), len(target)
    D = [[0] * (m + 1) for _ in range(n + 1)]
    # Khởi tạo hàng 0 và cột 0
    for i in range(1, n + 1):
        D[i][0] = D[i-1][0] + del_cost(source[i-1])
    for j in range(1, m + 1):
        D[0][j] = D[0][j-1] + ins_cost(target[j-1])
    # Truy hồi
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            D[i][j] = min(D[i-1][j]   + del_cost(source[i-1]),
                          D[i-1][j-1] + sub_cost(source[i-1], target[j-1]),
                          D[i][j-1]   + ins_cost(target[j-1]))
    return D

def min_edit_distance(source, target, **costs):
    return edit_matrix(source, target, **costs)[len(source)][len(target)]

print(min_edit_distance("intention", "execution"))   # kỳ vọng 8

## 4. Bảng khoảng cách `intention` ↔ `execution` (Fig. 2.20)

Sách vẽ bảng với hàng `#` ở **dưới cùng**; ở đây ta vẽ hàng `#` ở **trên cùng** cho dễ đọc. Ô có dấu `*` nằm trên một alignment tối ưu (xem phần 8).

In [ ]:
def print_matrix(source, target, D, path=()):
    path = set(path)
    cols = ["#"] + list(target)
    print("      " + "".join(f"{c:>5}" for c in cols))
    for i, row in enumerate(D):
        label = "#" if i == 0 else source[i-1]
        cells = "".join(f"{(str(v) + '*') if (i, j) in path else str(v):>5}" for j, v in enumerate(row))
        print(f"{label:>4}  {cells}")

D = edit_matrix("intention", "execution")
print_matrix("intention", "execution", D)
print("\nKhoảng cách =", D[9][9])

## 5. Tính tay một ô để hiểu công thức

Ví dụ ô `D[3, 3]` ứng với `"int"` → `"exe"`. Ta in ra ba ứng viên.

In [ ]:
def explain_cell(source, target, i, j, D):
    d = D[i-1][j] + 1
    s_cost = 0 if source[i-1] == target[j-1] else 2
    sub = D[i-1][j-1] + s_cost
    ins = D[i][j-1] + 1
    print(f"Ô D[{i},{j}]: '{source[:i]}' -> '{target[:j]}'")
    print(f"  xoá '{source[i-1]}'        : D[{i-1},{j}] + 1 = {D[i-1][j]} + 1 = {d}")
    print(f"  chèn '{target[j-1]}'        : D[{i},{j-1}] + 1 = {D[i][j-1]} + 1 = {ins}")
    print(f"  thay '{source[i-1]}'->'{target[j-1]}'   : D[{i-1},{j-1}] + {s_cost} = {D[i-1][j-1]} + {s_cost} = {sub}")
    print(f"  => min = {min(d, ins, sub)}  (bảng ghi {D[i][j]})")

explain_cell("intention", "execution", 3, 3, D)
print()
explain_cell("intention", "execution", 4, 3, D)   # 'e' == 'e' -> thay thế tốn 0

## 6. Các trường hợp biên

Một bài toán chuẩn phải xử lý đúng *mọi* trường hợp. Ta kiểm tra bằng bảng test (có `assert`).

In [ ]:
cases = [
    # (nguồn, đích, kỳ vọng, mô tả)
    ("", "", 0, "hai chuỗi rỗng"),
    ("abc", "", 3, "đích rỗng -> xoá hết (i lần xoá)"),
    ("", "abc", 3, "nguồn rỗng -> chèn hết (j lần chèn)"),
    ("abc", "abc", 0, "giống hệt nhau"),
    ("a", "b", 2, "1 ký tự khác nhau: thay = 2 (= xoá + chèn)"),
    ("a", "a", 0, "1 ký tự giống nhau"),
    ("abc", "ABC", 6, "phân biệt hoa/thường: 3 phép thay x 2"),
    ("ab", "ba", 2, "đảo vị trí: xoá a + chèn a = 2 (không có phép hoán vị)"),
    ("abcdef", "abcxyz", 6, "chung tiền tố 'abc', 3 ký tự khác: 3 thay x 2"),
    ("xyzabc", "uvwabc", 6, "chung hậu tố"),
    ("abc", "xyz", 6, "hoàn toàn khác, cùng độ dài"),
    ("a", "abcdef", 5, "đích chứa nguồn: chèn 5"),
    ("abcdef", "a", 5, "nguồn chứa đích: xoá 5"),
    ("aaaa", "aa", 2, "ký tự lặp: xoá 2"),
    ("intention", "execution", 8, "ví dụ của sách"),
    ("exention", "execution", 3, "chuỗi trung gian trong Fig. 2.19: thay n->c (2) + chèn u (1)"),
    ("xin chào", "xin chao", 2, "tiếng Việt có dấu: à ≠ a nên thay 1 lần = 2"),
]
ok = True
for s, t, exp, why in cases:
    got = min_edit_distance(s, t)
    flag = "✅" if got == exp else "❌"
    ok &= (got == exp)
    print(f"{flag} D({s!r:>12}, {t!r:<12}) = {got:>2} (kỳ vọng {exp:>2})  # {why}")
assert ok
print("\nTất cả trường hợp biên đều đúng 🎉")

**Tính chất rút ra** (cũng là cách tự kiểm tra bài làm):

1. `D(X, X) = 0`.
2. **Đối xứng** (khi ins = del): `D(X, Y) = D(Y, X)`.
3. **Bất đẳng thức tam giác**: `D(X, Z) ≤ D(X, Y) + D(Y, Z)`.
4. Chặn trên/dưới: `|n − m| · min(ins,del) ≤ D ≤ n·del + m·ins`.

In [ ]:
import random
random.seed(1)
def rnd(): return "".join(random.choice("abc") for _ in range(random.randint(0, 7)))
for _ in range(500):
    x, y, z = rnd(), rnd(), rnd()
    dxy, dyx = min_edit_distance(x, y), min_edit_distance(y, x)
    assert dxy == dyx                                              # đối xứng
    assert min_edit_distance(x, x) == 0                            # D(X,X)=0
    assert min_edit_distance(x, z) <= dxy + min_edit_distance(y, z)  # tam giác
    assert abs(len(x) - len(y)) <= dxy <= len(x) + len(y)          # chặn
print("Các tính chất đúng trên 500 bộ chuỗi ngẫu nhiên ✅")

## 7. Các biến thể chi phí

Sách nhắc tới hai phiên bản Levenshtein: thay thế tốn **1** hoặc tốn **2** (tương đương chèn + xoá). Thuật toán còn cho phép **trọng số tuỳ ý**, kể cả phụ thuộc từng chữ cái.

In [ ]:
pairs = [("intention", "execution"), ("kitten", "sitting"), ("sunday", "saturday"), ("ab", "ba")]
print(f"{'cặp':<28}{'sub=2':>7}{'sub=1':>7}")
for s, t in pairs:
    d2 = min_edit_distance(s, t)                        # mặc định 1-1-2
    d1 = min_edit_distance(s, t, **dict(zip(("ins_cost","del_cost","sub_cost"), uniform(1,1,1))))
    print(f"{s+' -> '+t:<28}{d2:>7}{d1:>7}")

In [ ]:
# Chi phí bất đối xứng: xoá đắt gấp 3 lần chèn
ic, dc, sc = uniform(ins=1, dele=3, sub=2)
print("abc -> ''   :", min_edit_distance("abc", "",    ins_cost=ic, del_cost=dc, sub_cost=sc), "(3 lần xoá x 3)")
print("'' -> abc   :", min_edit_distance("", "abc",    ins_cost=ic, del_cost=dc, sub_cost=sc), "(3 lần chèn x 1)")
# Khi xoá quá đắt, thay thế (2) rẻ hơn xoá+chèn (3+1=4):
print("a -> b      :", min_edit_distance("a", "b",     ins_cost=ic, del_cost=dc, sub_cost=sc))

# Khoảng cách KHÔNG còn đối xứng
print("D(ab,abcd) =", min_edit_distance("ab", "abcd", ins_cost=ic, del_cost=dc, sub_cost=sc),
      "| D(abcd,ab) =", min_edit_distance("abcd", "ab", ins_cost=ic, del_cost=dc, sub_cost=sc))

In [ ]:
# Chi phí theo chữ cái: thay thế giữa 2 phím LÂN CẬN trên bàn phím QWERTY rẻ hơn (vì dễ gõ nhầm)
rows = ["qwertyuiop", "asdfghjkl", "zxcvbnm"]
pos = {ch: (r, c) for r, row in enumerate(rows) for c, ch in enumerate(row)}

def adjacent(a, b):
    if a not in pos or b not in pos: return False
    (r1, c1), (r2, c2) = pos[a], pos[b]
    return abs(r1 - r2) <= 1 and abs(c1 - c2) <= 1

def keyboard_sub(a, b):
    if a == b: return 0
    return 1 if adjacent(a, b) else 2

print("wprd -> word  (p,o kề nhau):", min_edit_distance("wprd", "word", sub_cost=keyboard_sub))
print("wprd -> ward  (p,a xa nhau):", min_edit_distance("wprd", "ward", sub_cost=keyboard_sub))
print("Bản chuẩn 1-1-2 cho cả hai  :", min_edit_distance("wprd", "word"), min_edit_distance("wprd", "ward"))

## 8. Alignment: backpointer + backtrace (Fig. 2.22)

Biết *khoảng cách* chưa đủ; nhiều ứng dụng cần biết **ghép cặp nào**. Hai bước:

1. Khi điền ô, ghi **backpointer** (mũi tên) tới các ô đã dẫn tới giá trị nhỏ nhất. Một ô có thể có **nhiều** mũi tên khi có hoà.
   - `↖` đến từ chéo (giữ hoặc thay thế), `↑` đến từ ô phía trên (xoá), `←` đến từ ô bên trái (chèn).
2. **Backtrace**: từ ô `(n, m)` đi ngược theo mũi tên về `(0, 0)`. Mỗi đường đầy đủ là một **alignment tối ưu**.

In [ ]:
def backpointers(source, target, D, ins_cost=None, del_cost=None, sub_cost=None):
    d0 = uniform()
    ins_cost = ins_cost or d0[0]; del_cost = del_cost or d0[1]; sub_cost = sub_cost or d0[2]
    n, m = len(source), len(target)
    B = [[set() for _ in range(m + 1)] for _ in range(n + 1)]
    for i in range(n + 1):
        for j in range(m + 1):
            if i > 0 and D[i][j] == D[i-1][j] + del_cost(source[i-1]):               B[i][j].add("up")
            if j > 0 and D[i][j] == D[i][j-1] + ins_cost(target[j-1]):               B[i][j].add("left")
            if i > 0 and j > 0 and D[i][j] == D[i-1][j-1] + sub_cost(source[i-1], target[j-1]): B[i][j].add("diag")
    return B

ARROW = {"diag": "↖", "left": "←", "up": "↑"}
def print_pointers(source, target, D, B):
    cols = ["#"] + list(target)
    print("      " + "".join(f"{c:^9}" for c in cols))
    for i in range(len(D)):
        label = "#" if i == 0 else source[i-1]
        cells = ""
        for j in range(len(D[0])):
            arrows = "".join(ARROW[a] for a in ("diag", "left", "up") if a in B[i][j])
            cells += f"{arrows + str(D[i][j]):^9}"
        print(f"{label:>4}  {cells}")

D = edit_matrix("intention", "execution")
B = backpointers("intention", "execution", D)
print_pointers("intention", "execution", D, B)

Quan sát: nhiều ô có **2–3 mũi tên** (ví dụ `↖←↑5`). Đó chính là lý do có nhiều alignment tối ưu.

In [ ]:
def backtrace_one(source, target, D, B):
    """Đi ngược từ (n, m) theo 1 mũi tên (ưu tiên chéo > lên > trái). Trả về danh sách ops."""
    i, j, ops, cells = len(source), len(target), [], [(len(source), len(target))]
    while i > 0 or j > 0:
        if "diag" in B[i][j]:
            a, b = source[i-1], target[j-1]
            ops.append(("keep" if a == b else "sub", a, b)); i -= 1; j -= 1
        elif "up" in B[i][j]:
            ops.append(("del", source[i-1], None)); i -= 1
        else:
            ops.append(("ins", None, target[j-1])); j -= 1
        cells.append((i, j))
    return ops[::-1], cells

def show_alignment(ops):
    sym = {"keep": "|", "sub": "s", "del": "d", "ins": "i"}
    top = [(a if a is not None else "*") for _, a, _ in ops]
    bot = [(b if b is not None else "*") for _, _, b in ops]
    mid = [sym[o] for o, _, _ in ops]
    w = [max(len(x), len(y), 1) for x, y in zip(top, bot)]
    for line in (top, mid, bot):
        print(" ".join(c.center(k) for c, k in zip(line, w)))

ops, cells = backtrace_one("intention", "execution", D, B)
show_alignment(ops)
print("\nCác bước:", [o for o, _, _ in ops])
print_matrix("intention", "execution", D, path=cells)

Đọc alignment: `d` = xoá, `i` = chèn, `s` = thay thế, `|` = giữ nguyên. Dấu `*` là "khoảng trống" (gap).
Đúng như trong sách: **d s s i s** (các ký tự còn lại `e`, `t`, `i`, `o`, `n` được giữ nguyên).

Cách nhận biết trên đường đi qua bảng (Fig. 2.22): hai ô in đậm **cùng hàng** ⇒ có *chèn*; hai ô **cùng cột** ⇒ có *xoá*.

In [ ]:
def all_alignments(source, target, B, limit=10**6):
    """Liệt kê MỌI alignment tối ưu (mọi đường đi theo mũi tên)."""
    out = []
    def rec(i, j, acc):
        if len(out) >= limit: return
        if i == 0 and j == 0:
            out.append(acc[::-1]); return
        if "diag" in B[i][j]:
            a, b = source[i-1], target[j-1]
            rec(i-1, j-1, acc + [("keep" if a == b else "sub", a, b)])
        if "up" in B[i][j]:
            rec(i-1, j, acc + [("del", source[i-1], None)])
        if "left" in B[i][j]:
            rec(i, j-1, acc + [("ins", None, target[j-1])])
    rec(len(source), len(target), [])
    return out

alls = all_alignments("intention", "execution", B)
print("Số alignment tối ưu (chi phí 1-1-2):", len(alls))
print("Ba alignment đầu tiên:\n")
for k, ops_k in enumerate(alls[:3], 1):
    print(f"# {k}"); show_alignment(ops_k); print()

In [ ]:
# Với Levenshtein 1-1-1, số alignment tối ưu ít hơn nhiều (và khoảng cách = 5)
ic, dc, sc = uniform(1, 1, 1)
D1 = edit_matrix("intention", "execution", ic, dc, sc)
B1 = backpointers("intention", "execution", D1, ic, dc, sc)
alls1 = all_alignments("intention", "execution", B1)
print("Khoảng cách:", D1[9][9], "| số alignment tối ưu:", len(alls1))
show_alignment(alls1[0])

### Chi phí của một alignment

Tổng chi phí của alignment = tổng chi phí từng phép. Mọi alignment tìm được ở trên phải có tổng đúng bằng `D[n,m]`.

In [ ]:
def alignment_cost(ops, ins=1, dele=1, sub=2):
    price = {"keep": 0, "sub": sub, "del": dele, "ins": ins}
    return sum(price[o] for o, _, _ in ops)

assert all(alignment_cost(o) == 8 for o in alls)
print(f"Cả {len(alls)} alignment đều có chi phí 8 ✅")

from collections import Counter
print("Thống kê loại phép của alignment đầu tiên:", dict(Counter(o for o, _, _ in alls[0])))
print("Số kiểu (số phép ins/del/sub) khác nhau giữa các alignment:",
      sorted({tuple(sorted(Counter(o for o, _, _ in a).items())) for a in alls}))

## 9. Kiểm chứng tính đúng đắn

Hai cách kiểm tra độc lập:

1. **Áp dụng danh sách phép** lên chuỗi nguồn → phải ra đúng chuỗi đích.
2. **So với brute-force** (đệ quy có nhớ, viết độc lập) trên nhiều chuỗi ngẫu nhiên.

In [ ]:
def apply_ops(source, ops):
    out = []
    for o, a, b in ops:
        if o in ("keep", "sub", "ins"): out.append(b)   # đích nhận ký tự b
        # 'del' không thêm gì
    return "".join(out)

for s, t in [("intention", "execution"), ("kitten", "sitting"), ("", "abc"), ("abc", ""), ("same", "same")]:
    Dm = edit_matrix(s, t); Bm = backpointers(s, t, Dm)
    for ops_k in all_alignments(s, t, Bm):
        assert apply_ops(s, ops_k) == t
        assert "".join(a for o, a, _ in ops_k if a is not None) == s   # cột trên khôi phục nguồn
        assert alignment_cost(ops_k) == Dm[len(s)][len(t)]
print("Mọi alignment tái tạo đúng chuỗi đích và đúng chi phí ✅")

In [ ]:
from functools import lru_cache

def brute(s, t):
    @lru_cache(None)
    def f(a, b):
        if not a: return len(b)
        if not b: return len(a)
        sc = 0 if a[-1] == b[-1] else 2
        return min(f(a[:-1], b) + 1, f(a, b[:-1]) + 1, f(a[:-1], b[:-1]) + sc)
    return f(s, t)

random.seed(7)
for _ in range(1000):
    x = "".join(random.choice("abcd") for _ in range(random.randint(0, 8)))
    y = "".join(random.choice("abcd") for _ in range(random.randint(0, 8)))
    assert min_edit_distance(x, y) == brute(x, y)
print("DP khớp brute-force trên 1000 cặp ngẫu nhiên ✅")

## 10. Tối ưu hoá

### 10.1 Chỉ cần 2 hàng (bộ nhớ O(m)) — khi chỉ cần *khoảng cách*, không cần alignment
Mỗi ô chỉ phụ thuộc hàng trước và ô bên trái ⇒ giữ 2 hàng là đủ.

In [ ]:
def med_two_rows(s, t, ins=1, dele=1, sub=2):
    n, m = len(s), len(t)
    prev = [j * ins for j in range(m + 1)]
    for i in range(1, n + 1):
        cur = [prev[0] + dele] + [0] * m
        for j in range(1, m + 1):
            sc = 0 if s[i-1] == t[j-1] else sub
            cur[j] = min(prev[j] + dele, cur[j-1] + ins, prev[j-1] + sc)
        prev = cur
    return prev[m]

random.seed(3)
for _ in range(500):
    x = "".join(random.choice("abc") for _ in range(random.randint(0, 9)))
    y = "".join(random.choice("abc") for _ in range(random.randint(0, 9)))
    assert med_two_rows(x, y) == min_edit_distance(x, y)

a, b = "ab" * 600, "ba" * 600
t0 = time.time(); r = med_two_rows(a, b); print("Chuỗi 1200 ký tự: D =", r, f"({time.time()-t0:.2f}s, bộ nhớ ~2 hàng)")

### 10.2 Dừng sớm khi vượt ngưỡng k
Nếu ta chỉ quan tâm "hai chuỗi có cách nhau ≤ k không?" (ví dụ tìm từ gợi ý sửa lỗi), có thể dừng ngay khi **mọi** giá trị của một hàng đều > k (vì giá trị chỉ tăng dần theo hàng).

In [ ]:
def within_k(s, t, k, ins=1, dele=1, sub=2):
    """Trả về khoảng cách nếu <= k, ngược lại trả về None (có dừng sớm)."""
    if abs(len(s) - len(t)) * min(ins, dele) > k: return None
    prev = [j * ins for j in range(len(t) + 1)]
    for i in range(1, len(s) + 1):
        cur = [prev[0] + dele] + [0] * len(t)
        for j in range(1, len(t) + 1):
            sc = 0 if s[i-1] == t[j-1] else sub
            cur[j] = min(prev[j] + dele, cur[j-1] + ins, prev[j-1] + sc)
        if min(cur) > k: return None          # dừng sớm
        prev = cur
    return prev[-1] if prev[-1] <= k else None

print(within_k("intention", "execution", 8))   # 8
print(within_k("intention", "execution", 7))   # None
print(within_k("hello", "help", 3))            # 3 (xoá l + thay o->p = 1 + 2)

## 11. Ứng dụng

### 11.1 Gợi ý sửa lỗi chính tả
Chọn các từ trong từ điển có khoảng cách nhỏ nhất tới từ gõ sai.

In [ ]:
dictionary = ["word", "ward", "wood", "world", "sword", "work", "weird", "wore", "worm", "cord"]

def suggest(typo, words, k=3, **costs):
    scored = sorted((min_edit_distance(typo, w, **costs), w) for w in words)
    return scored[:k]

print("Chuẩn 1-1-2       :", suggest("wprd", dictionary))
print("Theo bàn phím     :", suggest("wprd", dictionary, sub_cost=keyboard_sub))

Với chi phí chuẩn, `word`/`ward`/`wood`... hoà nhau; trọng số theo bàn phím phân biệt được: `p` nằm cạnh `o` nên `word` là gợi ý hợp lý nhất.

### 11.2 Word Error Rate (WER) — nhận dạng giọng nói
Áp dụng cùng thuật toán nhưng **trên từ** thay vì ký tự, với chi phí 1-1-1:

`WER = (S + D + I) / N` (N = số từ của câu chuẩn).

In [ ]:
def wer(reference, hypothesis):
    ref, hyp = reference.split(), hypothesis.split()
    ic, dc, sc = uniform(1, 1, 1)
    D = edit_matrix(ref, hyp, ic, dc, sc)          # edit_matrix chạy được trên LIST từ
    B = backpointers(ref, hyp, D, ic, dc, sc)
    ops, _ = backtrace_one(ref, hyp, D, B)
    c = Counter(o for o, _, _ in ops)
    return D[len(ref)][len(hyp)] / len(ref), c, ops

ref = "the quick brown fox jumps over the lazy dog"
hyp = "the quick brown fox jump over lazy dog today"
rate, c, ops = wer(ref, hyp)
show_alignment(ops)
print(f"\nS={c['sub']}, D={c['del']}, I={c['ins']}, N={len(ref.split())}  =>  WER = {rate:.3f}")

### 11.3 Gợi ý về Viterbi: alignment **xác suất cực đại**
Cuối mục 2.9.1, sách nói Viterbi là mở rộng xác suất của min edit distance. Nếu ta đặt `cost = −log(xác suất)`, thì **tổng chi phí nhỏ nhất ⇔ tích xác suất lớn nhất**. Bản demo nhỏ bên dưới dùng đúng thuật toán trên:

In [ ]:
import math
# Xác suất giả định (ví dụ minh hoạ): giữ nguyên rất hay gặp, lỗi gõ ít gặp
P_MATCH, P_INS, P_DEL, P_SUB_NEAR, P_SUB_FAR = 0.90, 0.02, 0.02, 0.04, 0.005
nl = lambda p: -math.log(p)

ins_c = lambda c: nl(P_INS)
del_c = lambda c: nl(P_DEL)
sub_c = lambda a, b: nl(P_MATCH) if a == b else (nl(P_SUB_NEAR) if adjacent(a, b) else nl(P_SUB_FAR))

for w in ["word", "ward", "wood"]:
    cost = min_edit_distance("wprd", w, ins_cost=ins_c, del_cost=del_c, sub_cost=sub_c)
    print(f"wprd -> {w:<5} chi phí = {cost:6.3f}  |  xác suất alignment tốt nhất = {math.exp(-cost):.3e}")

## 12. Liên hệ với LCS (Longest Common Subsequence)

Khi **chèn = xoá = 1** và **thay thế = 2** (không bao giờ lợi hơn xoá + chèn), khoảng cách chỉ phụ thuộc vào phần chung dài nhất:

> `D(X, Y) = n + m − 2 · LCS(X, Y)`

In [ ]:
def lcs_len(a, b):
    L = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            L[i][j] = L[i-1][j-1] + 1 if a[i-1] == b[j-1] else max(L[i-1][j], L[i][j-1])
    return L[-1][-1]

for s, t in [("intention", "execution"), ("kitten", "sitting"), ("abc", "xyz")]:
    l = lcs_len(s, t)
    print(f"{s}/{t}: n+m-2*LCS = {len(s)+len(t)-2*l}  | D = {min_edit_distance(s, t)}")
random.seed(11)
for _ in range(500):
    x = "".join(random.choice("abc") for _ in range(random.randint(0, 8)))
    y = "".join(random.choice("abc") for _ in range(random.randint(0, 8)))
    assert min_edit_distance(x, y) == len(x) + len(y) - 2 * lcs_len(x, y)
print("Đẳng thức đúng trên 500 cặp ngẫu nhiên ✅")

## 📌 Tổng kết

- Edit distance = số phép (có trọng số) tối thiểu để biến chuỗi nguồn thành chuỗi đích.
- Quy hoạch động: lưu `D[i,j]`, mỗi ô chỉ cần 3 ô lân cận ⇒ **O(n·m)** thời gian.
- Thêm **backpointer** + **backtrace** ⇒ có **alignment** (có thể có nhiều alignment hoà nhau).
- Trọng số tuỳ ý cho phép mô hình hoá lỗi gõ bàn phím; chuyển sang `−log xác suất` ⇒ **Viterbi**.
- Ứng dụng: sửa chính tả, WER trong nhận dạng giọng nói, căn chỉnh câu trong dịch máy.

👉 Sang notebook **bài tập** để tự cài đặt và luyện tập.